# Conversion logic

> Module containing the reading and conversion logic

In [ ]:
#| default_exp convert

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import geopandas as gpd
import pandas as pd
import re
from pathlib import Path
from sonarlight import Sonar

## Conversion steps

We need to do several processing steps to go from the `sl2` or `sl3` data to a `csv` and `shape` we can use in our GIS software.

**From depth in meter to bottom height in mNAP**

To convert from depth measurement in meter to bottom height measurement in mNAP we need to know the height in mNAP from which the depth measurements were taken. Currently we add this height manually in the filename in cmNAP. Some examples:

- `2024-07-11_zuiderpark Hoogeveen2_+1075cmnap.sl2`
- `Sonar_2022-04-26_21.07.11beschrijving+0765cmNAP.sl3`

We extract this height from the filename.

The height of the Sonar boot is also stored in the `gps_altitude` column from the `sl2` and `sl3` files, but we haven't yet implemented the conversion from this height to mNAP height.

**Filter relevant facts**

We only need the facts that have the value "primary" in the column "survey".

**Transformation to the correct CRS**

The coördinates in the Sonar files are in crs WGS84 (epsg:4326), we need to convert those to the crs we use, which is RDN Amersfoort (epsg:28992).
We accomplish that by using the `geopandas` method `.set_crs` and `.to_crs`.

## Importing modules

We will use [sonarlight](https://github.com/KennethTM/sonarlight) to read the measurements from the `sl2` or `sl3` files.

::: {.callout-note}
Previously we used [sslib](https://github.com/opensounder/python-sllib) to parse the sonar files. But this package latest commit was 4 years ago. This `sslib` package also has less stars and the `sonarlight` has some neat extra features, such as simple conversion to a Pandas dataframe.
:::

## Load sl3 file for testing.

In [ ]:
sl3_f = Path("../test/Sonar_2022-04-26_21.07.11beschrijving+0765cmNAP.sl3")

In [ ]:
#| export
def read_sl(
    filepath: Path # The absolute location of the file to convert
    )->Sonar:
    return Sonar(str(filepath))

In [ ]:
sl3_d = read_sl(sl3_f)
sl3_d

Summary of SL3 file:

- Primary channel with 1320 frames
- Secondary channel with 1320 frames
- Downscan channel with 1320 frames
- Sidescan channel with 1319 frames

Start time: 2022-04-26 11:08:49.101999998
End time: 2022-04-26 11:10:19.315000057

File info: version 3, device 2, blocksize 3200, frame version 10

In [ ]:
sl3_d.df.columns

Index(['id', 'survey', 'datetime', 'x', 'y', 'longitude', 'latitude',
       'min_range', 'max_range', 'water_depth', 'gps_speed', 'gps_heading',
       'gps_altitude', 'bottom_index', 'frames'],
      dtype='str')

In [ ]:
sl3_d.df[['gps_altitude', 'water_depth']]

,gps_altitude,water_depth
1570,-1.91,0.606861
1573,-1.91,0.606861
1576,-1.91,0.606861
1577,-1.91,0.606861
1579,-1.91,0.609836
...,...,...
12838,-5.09,1.122443
12839,-5.09,1.122443
12840,-5.09,1.122443
12843,-5.09,1.122443


In [ ]:
type(sl3_d)

sonarlight.sonar_class.Sonar

In [ ]:
sl3_d.df.head()

,id,survey,datetime,x,y,longitude,latitude,min_range,max_range,water_depth,gps_speed,gps_heading,gps_altitude,bottom_index,frames
1570,174,primary,2022-04-26 11:08:49.101999998,674208,6913399,6.076888,52.748741,0.000000,36.576000,0.606861,0.127106,0.246756,-1.91,50,"[137, 137, 137, 137, 137, 129, 124, 119, 114, ..."
1573,174,secondary,2022-04-26 11:08:49.101999998,674208,6913399,6.076888,52.748741,0.000000,36.576000,0.606861,0.127106,0.246756,-1.91,50,"[137, 137, 137, 137, 137, 129, 124, 119, 114, ..."
1576,174,downscan,2022-04-26 11:08:49.239000082,674207,6913400,6.076879,52.748746,0.000000,21.945601,0.606861,0.127106,0.246756,-1.91,38,"[152, 152, 152, 152, 129, 143, 140, 137, 140, ..."
1577,352,sidescan,2022-04-26 11:08:49.240000010,674207,6913400,6.076879,52.748746,-39.989758,39.989758,0.606861,0.127106,0.246756,-1.91,21,"[42, 26, 41, 38, 43, 46, 43, 47, 50, 50, 53, 5..."
1579,175,primary,2022-04-26 11:08:49.249000072,674208,6913399,6.076888,52.748741,0.000000,3.992880,0.609836,0.122237,0.247439,-1.91,469,"[216, 216, 216, 216, 216, 216, 216, 216, 216, ..."


In [ ]:
sl3_d.bottom("primary")[0]

array([138, 136, 136, ...,  90,  92,  90], shape=(2061,), dtype=uint8)

In [ ]:
sl3_df = sl3_d.df

::: {.callout-tip collapse="true"}
## 🤖 AI chat: Notes to use `gps_altitude` column from `sl3`-file instead of extrachting the height of the measurement instrument from the filename

___
🤔 _The `sl3_df_sml` dataframe also has a column `gps_altitude`. Can you explain what this probably is. Given that longitude and latitude are given in the WGS84 coordinate system?_

## Convert Pandas DataFrame to GeoDataFrame

::: {.callout-tip collapse="true"}
## 🤖 AI chat: Pandas Dataframe to GeoDataFrame

___
🤔 _How to convert the pandas dataframe `sl2df_sml` to a geodataframe?_

In [ ]:
gdf = gpd.GeoDataFrame(sl3_df, geometry=gpd.points_from_xy(sl3_df.longitude, sl3_df.latitude))

In [ ]:
gdf = gdf.set_crs(epsg=4326)

In [ ]:
gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [ ]:
gdf = gdf.to_crs(epsg=28992)

In [ ]:
gdf.crs

<Projected CRS: EPSG:28992>
Name: Amersfoort / RD New
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: Netherlands - onshore, including Waddenzee, Dutch Wadden Islands and 12-mile offshore coastal zone.
- bounds: (3.2, 50.75, 7.22, 53.7)
Coordinate Operation:
- name: RD New
- method: Oblique Stereographic
Datum: Amersfoort
- Ellipsoid: Bessel 1841
- Prime Meridian: Greenwich

::: {.callout-tip collapse="true"}
## 🤖 AI regex chat

___
🤔 _I want to build a regex to extract the height in all of the following cases:

`measurements_-720cmnap.sl2`
`measurements_+720cmNap.sl3`
`measurements_-1720cmnap.sl3`
`measurements-1720cmnap.sl3`
`measurements+20cmNAP.sl2`
`measurements+20CMNAP.sl2`_

In [ ]:
rgx = r"[+-]\d+(?=cmnap)"

In [ ]:
int(re.search(rgx, str(sl3_f), flags=re.IGNORECASE)[0])

765

In [ ]:
#| export
def extract_height(
    sl_filepath: Path, # The absolute location of the file to convert
    re_ptrn: str=r"[+-]?\d+(?=cmnap)"
    )->int: # The height of the measurement station in cm above NAP
    "Extract height from the filename in cmNAP"
    if 'cmnap' not in str(sl_filepath).lower():
        raise ValueError("The filename must contain the height of the Sonar boot at time of measurement in 'cmNAP' at the end of the filename (e.g. 'example_description_+1050cmNAP.sl2')")
    return int(re.search(re_ptrn, str(sl_filepath), flags=re.IGNORECASE)[0])

Test function `extract_height`

In [ ]:
(extract_height(Path("/some/where/afen22e34_1823cmNAP.sl2")),
extract_height(Path("/some/where/afen22e34-1823cmNAP.sl2")),
extract_height(Path("/some/where/afen22e34br_-1823cmNAP.sl2")))

(1823, -1823, -1823)

In [ ]:
def slx2gdf(
    sl_filepath: Path, # The absolute location of the file to convert
    msrmnt_height: int, # Height of the measurement instrument at time of taking the measurements
    to_crs: str = "epsg:28992", # epsg code of crs to transform the coördinates to
    )->gpd:
    "Convert a sl2 or sl3 file to a GeoDataFrame with the given crs."
    s = Sonar(str(sl_filepath))
    df = s.df
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude))
    gdf = gdf.set_crs(epsg=4326)
    return gdf.to_crs(to_crs)

Test function `slx2gdf`

In [ ]:
sl3_gdf = slx2gdf(sl3_f, 1823)

In [ ]:
"water_depth" in gdf.columns

True

In [ ]:
sl3_gdf.head()

,id,survey,datetime,x,y,longitude,latitude,min_range,max_range,water_depth,gps_speed,gps_heading,gps_altitude,bottom_index,frames,geometry
1570,174,primary,2022-04-26 11:08:49.101999998,674208,6913399,6.076888,52.748741,0.000000,36.576000,0.606861,0.127106,0.246756,-1.91,50,"[137, 137, 137, 137, 137, 129, 124, 119, 114, ...",POINT (201568.299 529266.871)
1573,174,secondary,2022-04-26 11:08:49.101999998,674208,6913399,6.076888,52.748741,0.000000,36.576000,0.606861,0.127106,0.246756,-1.91,50,"[137, 137, 137, 137, 137, 129, 124, 119, 114, ...",POINT (201568.299 529266.871)
1576,174,downscan,2022-04-26 11:08:49.239000082,674207,6913400,6.076879,52.748746,0.000000,21.945601,0.606861,0.127106,0.246756,-1.91,38,"[152, 152, 152, 152, 129, 143, 140, 137, 140, ...",POINT (201567.685 529267.472)
1577,352,sidescan,2022-04-26 11:08:49.240000010,674207,6913400,6.076879,52.748746,-39.989758,39.989758,0.606861,0.127106,0.246756,-1.91,21,"[42, 26, 41, 38, 43, 46, 43, 47, 50, 50, 53, 5...",POINT (201567.685 529267.472)
1579,175,primary,2022-04-26 11:08:49.249000072,674208,6913399,6.076888,52.748741,0.000000,3.992880,0.609836,0.122237,0.247439,-1.91,469,"[216, 216, 216, 216, 216, 216, 216, 216, 216, ...",POINT (201568.299 529266.871)


## Building filter on GeoDataFrame

In [ ]:
sl3_gdf_pr = sl3_gdf[sl3_gdf["survey"]=="primary"]
sl3_gdf_pr.head()

,id,survey,datetime,x,y,longitude,latitude,min_range,max_range,water_depth,gps_speed,gps_heading,gps_altitude,bottom_index,frames,geometry
1570,174,primary,2022-04-26 11:08:49.101999998,674208,6913399,6.076888,52.748741,0.0,36.57600,0.606861,0.127106,0.246756,-1.91,50,"[137, 137, 137, 137, 137, 129, 124, 119, 114, ...",POINT (201568.299 529266.871)
1579,175,primary,2022-04-26 11:08:49.249000072,674208,6913399,6.076888,52.748741,0.0,3.99288,0.609836,0.122237,0.247439,-1.91,469,"[216, 216, 216, 216, 216, 216, 216, 216, 216, ...",POINT (201568.299 529266.871)
1587,176,primary,2022-04-26 11:08:49.290999889,674208,6913399,6.076888,52.748741,0.0,3.99288,0.609836,0.118954,0.248092,-1.91,469,"[216, 216, 216, 216, 216, 216, 216, 216, 216, ...",POINT (201568.299 529266.871)
1596,177,primary,2022-04-26 11:08:49.378000021,674208,6913399,6.076888,52.748741,0.0,3.99288,0.612811,0.114456,0.248715,-1.91,471,"[216, 216, 216, 216, 216, 216, 216, 216, 216, ...",POINT (201568.299 529266.871)
1604,178,primary,2022-04-26 11:08:49.423000097,674208,6913399,6.076888,52.748741,0.0,3.99288,0.612811,0.107427,0.249878,-1.91,471,"[216, 216, 216, 216, 216, 216, 216, 216, 216, ...",POINT (201568.299 529266.871)


::: {.callout-tip collapse="true"}
## 🤖 AI chat: pd.df a value is trying to be set on a copy of a slice from a DataFrame

___
🤔 _Please explain the DataFrame warning._

```python
sl2_gdf_pr['bottom_height'] = 12.53/100 - sl2_gdf_pr['water_depth']
```

```text
/app/data/.local/lib/python3.12/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
```

In [ ]:
sl3_gdf_pr = sl3_gdf[sl3_gdf["survey"]=="primary"].copy()

In [ ]:
sl3_gdf_pr['bottom_height'] = 12.53/100 - sl3_gdf_pr['water_depth']

In [ ]:
sl3_gdf_pr.head()

,id,survey,datetime,x,y,longitude,latitude,min_range,max_range,water_depth,gps_speed,gps_heading,gps_altitude,bottom_index,frames,geometry,bottom_height
1570,174,primary,2022-04-26 11:08:49.101999998,674208,6913399,6.076888,52.748741,0.0,36.57600,0.606861,0.127106,0.246756,-1.91,50,"[137, 137, 137, 137, 137, 129, 124, 119, 114, ...",POINT (201568.299 529266.871),-0.481561
1579,175,primary,2022-04-26 11:08:49.249000072,674208,6913399,6.076888,52.748741,0.0,3.99288,0.609836,0.122237,0.247439,-1.91,469,"[216, 216, 216, 216, 216, 216, 216, 216, 216, ...",POINT (201568.299 529266.871),-0.484536
1587,176,primary,2022-04-26 11:08:49.290999889,674208,6913399,6.076888,52.748741,0.0,3.99288,0.609836,0.118954,0.248092,-1.91,469,"[216, 216, 216, 216, 216, 216, 216, 216, 216, ...",POINT (201568.299 529266.871),-0.484536
1596,177,primary,2022-04-26 11:08:49.378000021,674208,6913399,6.076888,52.748741,0.0,3.99288,0.612811,0.114456,0.248715,-1.91,471,"[216, 216, 216, 216, 216, 216, 216, 216, 216, ...",POINT (201568.299 529266.871),-0.487511
1604,178,primary,2022-04-26 11:08:49.423000097,674208,6913399,6.076888,52.748741,0.0,3.99288,0.612811,0.107427,0.249878,-1.91,471,"[216, 216, 216, 216, 216, 216, 216, 216, 216, ...",POINT (201568.299 529266.871),-0.487511


::: {.callout-tip collapse="true"}
## 🤖AI chat: `latitude` and `longitude`

___
🤔 _How can I check the precision of the `latitude` and `longitude` columns?_

In [ ]:
sl3_gdf_pr['longitude'].iloc[0]

np.float64(6.076888166251876)

::: {.callout-tip collapse="true"}
## 🤖 AI Chat: Geometry column

___
🤔 _How can I check if the precision of the `geometry` column is the same?_

In [ ]:
sl3_gdf_pr['geometry'].iloc[0].x
sl3_gdf_pr['geometry'].iloc[0].y

529266.87106734

::: {.callout-tip collapse="true"}
## 🤖 AI chat: Check meaning of `bottom_index` column

___
🤔 _I think that all measurements with the same "bottom_index" also have the same longitude and latitude and water_depth. How can I check that assumption?_

In [ ]:
sl3_gdf[sl3_gdf["survey"]=="primary"].groupby('bottom_index')[['longitude', 'latitude', 'water_depth']].nunique()

,longitude,latitude,water_depth
bottom_index,,,
50,1,1,1
328,1,2,1
330,2,2,1
332,1,1,1
333,2,2,2
...,...,...,...
1001,1,1,1
1005,2,1,1
1007,1,1,1


In [ ]:
sl3_d.df[sl3_d.df["bottom_index"]==475]

,id,survey,datetime,x,y,longitude,latitude,min_range,max_range,water_depth,gps_speed,gps_heading,gps_altitude,bottom_index,frames
9536,1111,primary,2022-04-26 11:09:52.848999977,674198,6913407,6.076798,52.748784,0.0,3.99288,0.617507,0.162986,4.224975,-5.5,475,"[214, 214, 214, 214, 214, 214, 214, 214, 214, ..."
9539,1111,secondary,2022-04-26 11:09:52.848999977,674198,6913407,6.076798,52.748784,0.0,3.99288,0.617507,0.162986,4.224975,-5.5,475,"[214, 214, 214, 214, 214, 214, 214, 214, 214, ..."
9544,1112,primary,2022-04-26 11:09:52.894999981,674198,6913407,6.076798,52.748784,0.0,3.99288,0.617507,0.164213,4.223330,-5.5,475,"[214, 214, 214, 214, 214, 214, 214, 214, 214, ..."
9547,1112,secondary,2022-04-26 11:09:52.894999981,674198,6913407,6.076798,52.748784,0.0,3.99288,0.617507,0.164213,4.223330,-5.5,475,"[214, 214, 214, 214, 214, 214, 214, 214, 214, ..."


::: {.callout-tip collapse="true"}
## 🤖 AI chat: Further investigate `bottom_index`

___
🤔 _It is mostly true that the same bottom_index is the same location and depth. But not always. So we must filter the resulting table on same locations. Could we use something like `.unique` to only keep those points that have a unique `longitude`, `latitude` combination?_

In [ ]:
def clean_gdf(
    gdf: gpd.GeoDataFrame, # GeoDataFrame from sl2gdf
    msrmnt_height: int, # Height of measurement instrument in cm above NAP
    ) -> gpd.GeoDataFrame: # Cleaned GeoDataFrame with bottom_height column
    "Filter primary survey data, remove duplicates, and calculate bottom height in mNAP"
    gdf_primary = gdf[gdf["survey"] == "primary"].copy()
    gdf_unique = gdf_primary.drop_duplicates(subset=['longitude', 'latitude', 'water_depth'])
    gdf_unique.loc[:, 'bottom_height'] = msrmnt_height / 100 - gdf_unique['water_depth']
    return gdf_unique

In [ ]:
cln_gdf = clean_gdf(sl3_gdf, 1823)

::: {.callout-tip collapse="true"}
## `.drop_duplicates()` warning: trying to be set on a copy of a slice
___
🤔 _Why do I get this warning in the function `clean_gdf`?_

```python
cln_gdf = clean_gdf(sl4_gdf, 1823)
```

```text
/home/jelle/code/sonar2csv_shape/.venv/lib/python3.11/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
```

Test grouping of measurements based on location and creating several aggregations columns with different aggregation functions

In [ ]:
gdf_grpd = gdf.groupby(['longitude', 'latitude'], as_index=False).agg(
    mean_depth=pd.NamedAgg(column="water_depth", aggfunc="mean"),
    min_depth=pd.NamedAgg(column="water_depth", aggfunc="min"),
    max_depth=pd.NamedAgg(column="water_depth", aggfunc="max"),
    geometery=pd.NamedAgg(column="geometry", aggfunc="first"),
    datetime=pd.NamedAgg(column="datetime", aggfunc="mean")
)

In [ ]:
gdf_grpd.head()

,longitude,latitude,mean_depth,min_depth,max_depth,geometery,datetime
0,6.07678,52.748746,0.941204,0.932471,0.950051,POINT (201560.99 529267.408),2022-04-26 11:10:03.997620224
1,6.07678,52.748752,0.991693,0.947595,1.031126,POINT (201560.985 529268.016),2022-04-26 11:10:02.506777600
2,6.07678,52.748757,1.068617,1.042099,1.078859,POINT (201560.979 529268.623),2022-04-26 11:10:01.312600064
3,6.07678,52.748762,1.067053,1.051482,1.077630,POINT (201560.973 529269.23),2022-04-26 11:10:00.477333504
4,6.07678,52.748768,0.970447,0.867511,1.051482,POINT (201560.967 529269.837),2022-04-26 11:09:59.711977216


In [ ]:
def clean_gdf(
    gdf: gpd.GeoDataFrame, # GeoDataFrame from sl2gdf
    msrmnt_height: int, # Height of measurement instrument in cm above NAP
    ) -> gpd.GeoDataFrame: # Cleaned GeoDataFrame with bottom_height column
    "Filter primary survey data, remove duplicates, and calculate bottom height in mNAP"
    return (gdf[gdf["survey"]=="primary"]
            .drop_duplicates(subset=['longitude', 'latitude', 'water_depth'])
            .assign(bottom_height=lambda x: msrmnt_height / 100 - x['water_depth']))

In [ ]:
def clean_gdf(
    gdf: gpd.GeoDataFrame, # GeoDataFrame from sl2gdf
    msrmnt_height: int, # Height of measurement instrument in cm above NAP
    ) -> gpd.GeoDataFrame: # Cleaned GeoDataFrame with bottom_height column
    "Filter primary survey data, remove duplicates, and calculate bottom height in mNAP"
    return (gdf[gdf["survey"]=="primary"]
            .groupby(['longitude', 'latitude'], as_index=False).agg(
                mean_depth=pd.NamedAgg(column="water_depth", aggfunc="mean"),
                min_depth=pd.NamedAgg(column="water_depth", aggfunc="min"),
                max_depth=pd.NamedAgg(column="water_depth", aggfunc="max"),
                geometry=pd.NamedAgg(column="geometry", aggfunc="first"),
                datetime=pd.NamedAgg(column="datetime", aggfunc="mean")
            )
            .assign(bottom_height=lambda x: msrmnt_height / 100 - x['mean_depth']))

::: {.callout-tip collapse="true"}
## 🤖 AI chat: Aggregation of geodataframe returns a pandas dataframe
___
🤔 _When I use `clean_gdf` the returned object is a Pandas Dataframe, not a GeoDataFrame. This probably is the result of the `.groupby` method that uses the `agg` method. How can I solve this? I want a GeoDataFrame?_

In [ ]:
#| export
def clean_gdf(
    gdf: gpd.GeoDataFrame, # GeoDataFrame from sl2gdf
    msrmnt_height: int, # Height of measurement instrument in cm above NAP
    ) -> gpd.GeoDataFrame: # Cleaned GeoDataFrame with bottom_height column
    "Filter primary survey data, remove duplicates, and calculate bottom height in mNAP"
    df_cln = (gdf[gdf["survey"]=="primary"]
            .groupby(['longitude', 'latitude'], as_index=False).agg(
                mean_depth=pd.NamedAgg(column="water_depth", aggfunc="mean"),
                min_depth=pd.NamedAgg(column="water_depth", aggfunc="min"),
                max_depth=pd.NamedAgg(column="water_depth", aggfunc="max"),
                geometry=pd.NamedAgg(column="geometry", aggfunc="first"),
                datetime=pd.NamedAgg(column="datetime", aggfunc="mean")
            )
            .assign(bottom_height=lambda x: msrmnt_height / 100 - x['mean_depth']))
    return gpd.GeoDataFrame(df_cln, geometry='geometry', crs=gdf.crs)

In [ ]:
cln_gdf = clean_gdf(sl3_gdf, 1823)

In [ ]:
type(cln_gdf)

geopandas.geodataframe.GeoDataFrame

In [ ]:
cln_gdf.head()

,longitude,latitude,mean_depth,min_depth,max_depth,geometry,datetime,bottom_height
0,6.076780,52.748757,1.073326,1.069168,1.078859,POINT (201560.979 529268.623),2022-04-26 11:10:01.221999872,17.156673
1,6.076780,52.748762,1.068734,1.051482,1.077630,POINT (201560.973 529269.23),2022-04-26 11:10:00.517230848,17.161266
2,6.076780,52.748768,0.969534,0.867511,1.051482,POINT (201560.967 529269.837),2022-04-26 11:09:59.692999936,17.260466
3,6.076780,52.748773,0.861626,0.858684,0.867511,POINT (201560.961 529270.444),2022-04-26 11:09:59.201000192,17.368374
4,6.076789,52.748752,0.958502,0.932471,0.999739,POINT (201561.593 529268.021),2022-04-26 11:10:02.960588288,17.271498


In [ ]:
cln_gdf['geometry'].iloc[0].x

201560.97882739094

::: {.callout-tip collapse="true"}
## 🤖 AI chat: Checking for needed columns
___
🤔 _I also would like to check if the Dataframe that is created from the sl2 or sl3 file contains the needed columns in the needed datatype. Should I write a function to check that? Should I add the checks within an existing function? Should I use Pydantic?_

In [ ]:
#| export
def slx2gdf(
    sl_filepath: Path, # The absolute location of the file to convert
    to_crs: str = "epsg:28992", # epsg code of crs to transform the coördinates to
    survey_fltr: str = "primary", # Filter measurement facts on survey value
    )->gpd:
    "Convert a sl2 or sl3 file to a GeoDataFrame with the given crs."
    s = Sonar(str(sl_filepath))
    df = s.df
    required_cols = ['longitude', 'latitude', 'water_depth', 'survey']
    if not all(col in df.columns for col in required_cols):
        raise KeyError(f"Missing one or more of the required columns in the converted sl2 or sl3 file.\nRequired columns are: 'longitude', 'latitude', 'water_depth' and 'survey'")
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude))
    gdf = gdf.set_crs(epsg=4326)
    return gdf.to_crs(to_crs)

In [ ]:
gdf = slx2gdf(sl3_f)
gdf.head()

,id,survey,datetime,x,y,longitude,latitude,min_range,max_range,water_depth,gps_speed,gps_heading,gps_altitude,bottom_index,frames,geometry
1570,174,primary,2022-04-26 11:08:49.101999998,674208,6913399,6.076888,52.748741,0.000000,36.576000,0.606861,0.127106,0.246756,-1.91,50,"[137, 137, 137, 137, 137, 129, 124, 119, 114, ...",POINT (201568.299 529266.871)
1573,174,secondary,2022-04-26 11:08:49.101999998,674208,6913399,6.076888,52.748741,0.000000,36.576000,0.606861,0.127106,0.246756,-1.91,50,"[137, 137, 137, 137, 137, 129, 124, 119, 114, ...",POINT (201568.299 529266.871)
1576,174,downscan,2022-04-26 11:08:49.239000082,674207,6913400,6.076879,52.748746,0.000000,21.945601,0.606861,0.127106,0.246756,-1.91,38,"[152, 152, 152, 152, 129, 143, 140, 137, 140, ...",POINT (201567.685 529267.472)
1577,352,sidescan,2022-04-26 11:08:49.240000010,674207,6913400,6.076879,52.748746,-39.989758,39.989758,0.606861,0.127106,0.246756,-1.91,21,"[42, 26, 41, 38, 43, 46, 43, 47, 50, 50, 53, 5...",POINT (201567.685 529267.472)
1579,175,primary,2022-04-26 11:08:49.249000072,674208,6913399,6.076888,52.748741,0.000000,3.992880,0.609836,0.122237,0.247439,-1.91,469,"[216, 216, 216, 216, 216, 216, 216, 216, 216, ...",POINT (201568.299 529266.871)


In [ ]:
gdf.crs

<Projected CRS: EPSG:28992>
Name: Amersfoort / RD New
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: Netherlands - onshore, including Waddenzee, Dutch Wadden Islands and 12-mile offshore coastal zone.
- bounds: (3.2, 50.75, 7.22, 53.7)
Coordinate Operation:
- name: RD New
- method: Oblique Stereographic
Datum: Amersfoort
- Ellipsoid: Bessel 1841
- Prime Meridian: Greenwich

In [ ]:
gdf['geometry'].iloc[0].x

201568.2991979634

::: {.callout-tip collapse="true"}
## 🤖 AI chat: Keep x and y coordinates in csv export
___
🤔 _I don't want to drop the geometry column completely. I want to keep the x and y coordinates in a x and y column. How do I do that?_

In [ ]:
#| export
def export_gdf(
    gdf: gpd.GeoDataFrame, # GeoDataFrame to be saved
    fn: str, # Filename of the GeoDataFrame without extension
    folder_out: Path, # Absolute path to folder where files can be saved
    esri_shp: bool=True, # Save GeoDataFrame to Esri shapefile?
    csv: bool=True, # Save GeoDataFrame to comma seperated file?
    geopckg: bool=True, # Save GeoDataFrame to geopackage?
    ) -> None:
    "Deze functie doet iets"
    if esri_shp: gdf.to_file(folder_out / f"{fn}.shp")
    if geopckg: gdf.to_file(folder_out / f"{fn}.gpkg", driver="GPKG")
    if csv:
        gdf['x'] = gdf.geometry.x
        gdf['y'] = gdf.geometry.y
        df = gdf.drop(columns=['geometry'])
        df.to_csv(folder_out / f"{fn}.csv", index=False)
    return

In [ ]:
sl3_f.stem

'Sonar_2022-04-26_21.07.11beschrijving+0765cmNAP'

In [ ]:
export_gdf(cln_gdf, sl3_f.stem, Path("../test/"))

/tmp/ipykernel_447/1576217988.py:10: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  if esri_shp: gdf.to_file(folder_out / f"{fn}.shp")
/app/data/.local/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field datetime create as date field, though DateTime requested.
  ogr_write(
/app/data/.local/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'bottom_height' to 'bottom_hei'
  ogr_write(


In [ ]:
#| export
def process_sonar_file(
    sl_filepath: Path, # Path to sl2 or sl3 file
    folder_out: Path, # Output folder for exported files
    to_crs: str = "epsg:28992" # Target CRS
    ) -> None:
    "Process sonar file: extract height, convert to GeoDataFrame, clean, and export"
    msrmnt_height = extract_height(str(sl_filepath.stem))
    gdf = slx2gdf(sl_filepath, to_crs=to_crs)
    gdf_clean = clean_gdf(gdf, msrmnt_height)
    export_gdf(gdf_clean, sl_filepath.stem, folder_out)

In [ ]:
process_sonar_file(sl3_f, Path("../test"))

/tmp/ipykernel_447/1576217988.py:10: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  if esri_shp: gdf.to_file(folder_out / f"{fn}.shp")
/app/data/.local/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field datetime create as date field, though DateTime requested.
  ogr_write(
/app/data/.local/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'bottom_height' to 'bottom_hei'
  ogr_write(


::: {.callout-tip collapse="true"}
## 🤖 AI chat: Difference between `gdf.loc[:, 'y'] =` and `gdf['y'] = `?
___
🤔 _Can you explain what the difference is "under the hood" between `gdf.loc[:, 'y'] =` and just `gdf['y'] = `?_

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()